# Movie Booking Conversational AI Notebook

This notebook mirrors the scripted project and provides an interactive, sectioned workflow for dataset generation, preprocessing, encoder training, simulated LLM prompting, evaluation, visualization, testing, and export.


In [8]:
# Setup and dependency check
try:
    import matplotlib
    import seaborn
    import pandas
    import numpy
    import torch
    print("Dependency check passed.")
    print("matplotlib:", matplotlib.__version__)
    print("seaborn:", seaborn.__version__)
    print("pandas:", pandas.__version__)
    print("numpy:", numpy.__version__)
    print("torch:", torch.__version__)
except Exception as exc:
    print("Missing dependency:", exc)
    print("If needed, run: pip install matplotlib seaborn pandas numpy torch")


Dependency check passed.
matplotlib: 3.10.9
seaborn: 0.13.2
pandas: 3.0.3
numpy: 2.4.6
torch: 2.12.0+cpu


In [9]:
# Import libraries and project modules
import json
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from llm.prompting import SimulatedLLMPipeline, build_prompt
from main import (
    build_artifacts,
    build_tokenizer,
    error_analysis,
    evaluate_encoder_model,
    evaluate_llm_pipeline,
    generate_adversarial_samples,
    load_or_generate_dataset,
    metric_comparison_figure,
    plot_dataset_statistics,
    save_predictions_examples,
    train_encoder_model,
)
from utils.preprocessing import build_label_maps, encode_dataframe, set_seed, split_dataset

print("Imports loaded.")


Imports loaded.


In [10]:
# Global configuration for interactive runs
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
SEED = 42
DEFAULT_EPOCHS = 4

print({"PROJECT_ROOT": str(PROJECT_ROOT), "SEED": SEED, "DEFAULT_EPOCHS": DEFAULT_EPOCHS})


{'PROJECT_ROOT': 'd:\\GIT\\ConversationalAI', 'SEED': 42, 'DEFAULT_EPOCHS': 4}


In [11]:
# Display and inspect main.py source
from IPython.display import Markdown, display
from pathlib import Path

main_source = Path("main.py").read_text(encoding="utf-8")
display(Markdown("## main.py source preview"))
print(main_source[:6000])


## main.py source preview

from __future__ import annotations

import argparse
import json
import random
import time
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from llm.prompting import SimulatedLLMPipeline, build_prompt, spans_from_entities
from models.encoder_model import MovieBookingEncoder
from utils.metrics import classification_metrics, confusion_matrix, entity_metrics, model_size_mb, summarize_latencies
from utils.preprocessing import (
    ENTITY_BANK,
    ENTITY_LABELS,
    INTENTS,
    align_tags_to_subwords,
    build_dataloader,
    build_label_maps,
    compute_basic_vocab,
    compute_oov_rate,
    dataset_statistics,
    deserialize_tokens,
    deserialize_tags,
    encode_dataframe,
    entity_distribution,
    gener

In [12]:
# Refactor script entrypoint into notebook-friendly functions
from types import SimpleNamespace


def parse_args(overrides=None):
    defaults = {"phase": "all", "epochs": 4}
    if overrides:
        defaults.update(overrides)
    return SimpleNamespace(**defaults)


def load_data():
    return load_or_generate_dataset()


def preprocess(dataset, train_df, val_df, test_df):
    return build_artifacts(dataset, train_df, val_df, test_df)


def run_workflow(args=None):
    return None

print("Notebook wrapper functions defined.")


Notebook wrapper functions defined.


In [15]:
# Load data and build preprocessing artifacts
dataset, train_df, val_df, test_df = load_or_generate_dataset()
tokenizer = build_tokenizer(train_df)
intent_to_id, tag_to_id, id_to_intent, id_to_tag = build_label_maps(dataset)
encoded_train = encode_dataframe(train_df, tokenizer, intent_to_id, tag_to_id)
encoded_val = encode_dataframe(val_df, tokenizer, intent_to_id, tag_to_id)
encoded_test = encode_dataframe(test_df, tokenizer, intent_to_id, tag_to_id)

print("Dataset shape:", dataset.shape)
print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Tokenizer vocab size:", len(tokenizer.vocab))


Dataset shape: (1000, 4)
Split sizes: 704 144 152
Tokenizer vocab size: 336


In [16]:
# Data loading and preprocessing
train_df, val_df, test_df = split_dataset(dataset, seed=42)
tokenizer = build_tokenizer(train_df)
intent_to_id, tag_to_id, id_to_intent, id_to_tag = build_label_maps(dataset)
encoded_train = encode_dataframe(train_df, tokenizer, intent_to_id, tag_to_id)
encoded_val = encode_dataframe(val_df, tokenizer, intent_to_id, tag_to_id)
encoded_test = encode_dataframe(test_df, tokenizer, intent_to_id, tag_to_id)

print(train_df.head(2).to_dict(orient="records"))
print("Tokenizer vocab size:", len(tokenizer.vocab))


[{'sentence': 'tell me the timngs for oppenheimer at inox city center', 'intent': 'check_showtime', 'tokens': '["tell", "me", "the", "timngs", "for", "oppenheimer", "at", "inox", "city", "center"]', 'BIO_tags': '["O", "O", "O", "O", "O", "B-MOVIE_NAME", "O", "B-THEATER_NAME", "I-THEATER_NAME", "I-THEATER_NAME"]'}, {'sentence': 'hey book 1 balcony tickets for oppenheimer at pvr nexus mall', 'intent': 'book_ticket', 'tokens': '["hey", "book", "1", "balcony", "tickets", "for", "oppenheimer", "at", "pvr", "nexus", "mall"]', 'BIO_tags': '["O", "O", "B-NUM_TICKETS", "B-SEAT_TYPE", "O", "O", "B-MOVIE_NAME", "O", "B-THEATER_NAME", "I-THEATER_NAME", "I-THEATER_NAME"]'}]
Tokenizer vocab size: 336


In [17]:
# Core processing / model training
model, history = train_encoder_model(tokenizer, intent_to_id, tag_to_id, encoded_train, encoded_val, epochs=2)
print("Training history:", history)


Training history: {'train_loss': [3.4771625995635986, 1.8883531039411372], 'val_loss': [2.429193046357897, 1.0621392130851746]}


In [18]:
# Evaluation and visualization
encoder_summary = evaluate_encoder_model(model, encoded_test, id_to_intent, id_to_tag)
llm_summary_by_strategy = evaluate_llm_pipeline(test_df)
structured_summary = llm_summary_by_strategy["structured_json"]

print("Encoder intent accuracy:", encoder_summary["intent_metrics"]["accuracy"])
print("Encoder entity strict F1:", encoder_summary["entity_metrics"]["strict_f1"])
print("LLM intent accuracy:", structured_summary["intent_metrics"]["accuracy"])
print("LLM entity strict F1:", structured_summary["entity_metrics"]["strict_f1"])


Encoder intent accuracy: 0.9342105263157895
Encoder entity strict F1: 0.3028720626631854
LLM intent accuracy: 0.5460526315789473
LLM entity strict F1: 0.8647342995169083


In [19]:
# Notebook-only helper for the sample prompt demo
pipeline = SimulatedLLMPipeline()
print("Simulated LLM pipeline ready.")


Simulated LLM pipeline ready.


In [20]:
# Small runnable example for quick validation
sample_sentence = "book 2 vip tickets for dune part two at pvr nexus mall"
llm_sample = pipeline.predict(sample_sentence, strategy="structured_json")
print("Prompt:\n", build_prompt(sample_sentence, strategy="structured_json"))
print("Parsed output:\n", json.dumps(llm_sample.parsed_output, indent=2))


Prompt:
 You are a movie booking assistant. Classify the user's intent and extract entities. Return a JSON object with keys intent and entities. Use the exact schema: {"intent": string, "entities": {label: value}}. User: book 2 vip tickets for dune part two at pvr nexus mall
Assistant:
Parsed output:
 {
  "intent": "check_showtime",
  "entities": {
    "MOVIE_NAME": "Dune Part Two",
    "THEATER_NAME": "PVR Nexus Mall",
    "SEAT_TYPE": "vip",
    "NUM_TICKETS": "2"
  }
}


In [21]:
# Lightweight in-notebook checks
assert isinstance(dataset, pd.DataFrame)
assert len(dataset) >= 1000
assert set(dataset.columns) >= {"sentence", "intent", "tokens", "BIO_tags"}
assert tokenizer.vocab
assert len(encoded_train) > 0
assert "structured_json" in llm_summary_by_strategy
print("Notebook assertions passed.")


Notebook assertions passed.


In [22]:
import json
from pathlib import Path

results_dir = Path("results")
artifacts = [
    results_dir / "encoder_summary.json",
    results_dir / "llm_summary.json",
    results_dir / "metric_comparison.png",
    results_dir / "encoder_confusion_matrix.png",
    results_dir / "llm_confusion_matrix.png",
    results_dir / "error_analysis.json",
]

for artifact in artifacts:
    print(f"{artifact}: {'exists' if artifact.exists() else 'missing'}")


results\encoder_summary.json: exists
results\llm_summary.json: exists
results\metric_comparison.png: exists
results\encoder_confusion_matrix.png: exists
results\llm_confusion_matrix.png: exists
results\error_analysis.json: exists


In [23]:
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd()
main_script = project_root / "main.py"

print("Running main.py from notebook...")
result = subprocess.run([sys.executable, str(main_script), "--phase", "generate"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)


Running main.py from notebook...
Generated dataset with 1000 examples at D:\GIT\ConversationalAI\data\dataset.csv


Return code: 0
